[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LizbethMG-Teaching/pose2behav-book/blob/main/notebooks/analysis_multi-animal.ipynb)

# 📓 Notebook 3 – Analysis of multi animal (top-view mouse)
## 1. Introduction & objectives

In this notebook, you will analyze pose estimation outputs generated with the SuperAnimal ModelZoo on a top view multi animal video containing several mice.

**Learning goals:**

After this notebook, you should be able to:
- Load and preprocess multi animal pose data from SuperAnimal DLC
- Implement your own filtering and interpolation choices
- Compute activity and social metrics per mouse
- Integrate the results into a summary table

--- 

**About this notebook**

In this notebook, you will analyze pose-estimation data from freely-moving mice. 

# 🐭🐭🏠🎥 The Mouse House: multi animal pose challenge

<img src="https://raw.githubusercontent.com/LizbethMG-Teaching/pose2behav-book/main/assets/illustrations/cover-mice.png" width="50%">

Five mice live together in the "Mouse House" 🐭🤍🏠, a fully monitored arena.
Every movement is tracked with SuperAnimal DeepLabCut.

Your task is to use pose data to build a behavioral profile for each mouse:
- Who is the Hyperactive One?
- Who is the Social Butterfly?
- Who is the Lone Wolf?
- And who wins each "medal" category?

🥇🥈🥉 At the end, you will assign gold, silver, and bronze medals in:
- Activity
- Sociability

For this exercise, you will work mostly independently, but everyone must create the same output variable names and structure so we can compare results.



--- 
**Instructions**

This notebook mixes pre-filled code cells (ready to run) and coding exercises that you will complete.

- Some cells are already complete (just run them).
- You are free to choose methods, but you must respect:
  - Input: the provided pose file
  - Output variable names and column names as indicated. 

👉 Here’s how to work through it:
1. Read carefully each section before running the cells.
2. When a cell requires you to code, you’ll see a TODO comment.

⚡ After finishing the course, feel free to experiment and modify the notebook as you like!


---

<img src="https://raw.githubusercontent.com/LizbethMG-Teaching/pose2behav-book/main/assets/single-frame-multi.png" width="50%">

**‼️ Useful reference information for this LAB**

- Frame resolution: 652 × 636 pixels
- Frame rate: 66 frames per second
- Mouse body length (nose to base of tail): 80 pixels
- Real body length: 9 cm
- Approximate scale: 1 cm ≈ 8.89 pixels
- Pixel to centimeter conversion: 1 pixel ≈ 0.1125 cm

Arena geometry

- Arena shape: circular
- Diameter in the image: 460 pixels
- Real-world diameter: about 52 cm
- For simplified calculations, the arena can be approximated as a 460 × 460 pixel square
- Upper left corner of this square: x = 108, y = −78 (image coordinate system)

## 2. Data Loading & Format Inspection

### 2.1 Download data (prefilled)

**📋 Instructions:**
- Run the code cell below to download the dataset file.

In [ ]:
# PREFILLED, NO NEED TO CHANGE, JUST RUN THIS CELL
# Install and import the required libraries:
!pip -q install gdown tables

import os
from pathlib import Path
import gdown, pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import groupby
import re
import numpy as np
from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle


# --------------------------------------------------------------

# Detect if running in Google Colab
if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ:
    DEST = Path("/content/cleaned_pose_downloaded.h5")
else:
    DEST = Path("cleaned_pose_downloaded.h5")  # save in current folder locally
print("Saving to:", DEST)

# Select here the experiment you want to download, comment the others:
# mice-5_5min

# File : "cleaned_pose_3mice_5min.h5"
FILE_ID = "1BxdhvhPl-2Yb39v1_ZICi9fzp_wv8u8q"

URL = f"https://drive.google.com/uc?id={FILE_ID}"

print("Downloading from Drive...")
_ = gdown.download(URL, str(DEST), quiet=False)

# Basic checks
assert DEST.exists() and DEST.stat().st_size > 0, "❌ Download failed or empty file."
print(f"✅ Downloaded to {DEST} ({DEST.stat().st_size/1_000_000:.2f} MB)")

# --- Load the cleaned H5 file into a pandas DataFrame ---
df = pd.read_hdf(DEST, key="df_with_missing")

print("✅ Data loaded successfully!")
print("Shape:", df.shape)
print("Columns:", list(df.columns)[:8], "...")

In [ ]:
# 👉🏼 PREFILLED CELL — JUST RUN IT

# Show the first few columns
print("\nFirst 10 columns:")
print(df.columns[:10])

# List animals present in this file
animals = sorted(set(col.split("_")[0] for col in df.columns))
print("\nAnimals in this file:", animals)

# Show how many columns each mouse has
print("\nNumber of columns per mouse:")
for a in animals:
    count = sum(col.startswith(a + "_") for col in df.columns)
    print(f"{a}: {count} columns")

# Function to extract a single mouse
def get_mouse(df, animal_id):
    """Returns a DataFrame with only one mouse's data and clean column names."""
    cols = [c for c in df.columns if c.startswith(f"{animal_id}_")]
    dfa = df[cols].copy()
    dfa.columns = [c.replace(f"{animal_id}_", "") for c in cols]
    return dfa

# Example: extract the first mouse
example_mouse = animals[0]
mouse_df = get_mouse(df, example_mouse)

print(f"\nExample: data for {example_mouse}")
print(mouse_df.head())

# Show NaN percentages for this mouse
print("\nPercent of missing values for this mouse:")
print(mouse_df.isna().mean() * 100)